In [2]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q sentencepiece protobuf
!pip install -q pillow matplotlib seaborn pandas numpy tqdm scikit-learn
!pip install -U transformers
!pip install -U datasets
!pip install -U peft
!pip install -U trl
!pip install -U accelerate
!pip install -U bitsandbytes
!pip install -U pillow
!pip install -U sentencepiece
!pip install -U qwen-vl-utils

In [3]:
# ============================================================
# Dataset Configuration
# ============================================================

import os
from pathlib import Path

# Root folder
DATA_ROOT = Path("processed_dataset")

# Train
TRAIN_IMAGE_DIR = DATA_ROOT / "train" / "images"
TRAIN_REPORT_DIR = DATA_ROOT / "train" / "reports"

# Validation
VAL_IMAGE_DIR = DATA_ROOT / "validation" / "images"
VAL_REPORT_DIR = DATA_ROOT / "validation" / "reports"

# Test
TEST_IMAGE_DIR = DATA_ROOT / "test" / "images"
TEST_REPORT_DIR = DATA_ROOT / "test" / "reports"

print(DATA_ROOT)

processed_dataset


In [4]:
# ============================================================
# Verify Dataset Structure
# ============================================================

print("Train Images     :", TRAIN_IMAGE_DIR.exists())
print("Train Reports    :", TRAIN_REPORT_DIR.exists())

print("Validation Images:", VAL_IMAGE_DIR.exists())
print("Validation Reports:", VAL_REPORT_DIR.exists())

print("Test Images      :", TEST_IMAGE_DIR.exists())
print("Test Reports     :", TEST_REPORT_DIR.exists())

Train Images     : True
Train Reports    : True
Validation Images: True
Validation Reports: True
Test Images      : True
Test Reports     : True


In [5]:
# ============================================================
# Count Dataset Files
# ============================================================

print("Training Images    :", len(list(TRAIN_IMAGE_DIR.glob("*.png"))))
print("Training Reports   :", len(list(TRAIN_REPORT_DIR.glob("*.txt"))))

print("Validation Images  :", len(list(VAL_IMAGE_DIR.glob("*.png"))))
print("Validation Reports :", len(list(VAL_REPORT_DIR.glob("*.txt"))))

print("Test Images        :", len(list(TEST_IMAGE_DIR.glob("*.png"))))
print("Test Reports       :", len(list(TEST_REPORT_DIR.glob("*.txt"))))

Training Images    : 21443
Training Reports   : 21443
Validation Images  : 4594
Validation Reports : 4594
Test Images        : 4596
Test Reports       : 4596


In [6]:
# ============================================================
# Dataset Libraries
# ============================================================

import os
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

In [7]:
# ============================================================
# Image Transform
# ============================================================

image_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor()

])

print("Image Transform Ready")

Image Transform Ready


In [8]:
# ============================================================
# Custom Chest X-ray Dataset
# ============================================================

class ChestXrayDataset(Dataset):

    def __init__(self, image_dir, report_dir, transform=None):

        self.image_dir = image_dir
        self.report_dir = report_dir
        self.transform = transform

        self.image_files = sorted(
            [file for file in os.listdir(image_dir) if file.endswith(".png")]
        )

        self.report_files = sorted(
            [file for file in os.listdir(report_dir) if file.endswith(".txt")]
        )

        assert len(self.image_files) == len(self.report_files), \
            "Number of images and reports do not match."

    def __len__(self):

        return len(self.image_files)

    def __getitem__(self, index):

        image_path = self.image_dir / self.image_files[index]
        report_path = self.report_dir / self.report_files[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:

            image = self.transform(image)

        with open(report_path, "r", encoding="utf-8") as file:

            report = file.read().strip()

        return {

            "image": image,

            "report": report,

            "image_path": str(image_path),

            "report_path": str(report_path)

        }

print("Dataset Class Created")

Dataset Class Created


In [9]:
# ============================================================
# Dataset Objects
# ============================================================

train_dataset = ChestXrayDataset(

    TRAIN_IMAGE_DIR,

    TRAIN_REPORT_DIR,

    transform=image_transform

)

validation_dataset = ChestXrayDataset(

    VAL_IMAGE_DIR,

    VAL_REPORT_DIR,

    transform=image_transform

)

test_dataset = ChestXrayDataset(

    TEST_IMAGE_DIR,

    TEST_REPORT_DIR,

    transform=image_transform

)

print("=" * 60)

print("Training Samples   :", len(train_dataset))

print("Validation Samples :", len(validation_dataset))

print("Test Samples       :", len(test_dataset))

print("=" * 60)

Training Samples   : 21443
Validation Samples : 4594
Test Samples       : 4596


In [11]:
# ============================================================
# Create DataLoaders
# ============================================================

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("=" * 60)
print("DataLoaders Created Successfully")
print("=" * 60)

DataLoaders Created Successfully


In [12]:
# ============================================================
# Verify DataLoader
# ============================================================

batch = next(iter(train_loader))

print("=" * 60)
print("Batch Loaded Successfully")
print("=" * 60)

print("Image Tensor Shape :", batch["image"].shape)
print("Number of Reports  :", len(batch["report"]))

print("\nSample Report:\n")
print(batch["report"][0][:500])

print("\nImage dtype :", batch["image"].dtype)

Batch Loaded Successfully
Image Tensor Shape : torch.Size([8, 3, 224, 224])
Number of Reports  : 8

Sample Report:

Patient is status post median sternotomy and CABG. Normal postoperative cardiomediastinal silhouette is stable. No focal consolidations, pleural effusions, pulmonary edema, or pneumothorax are seen. Right-sided PICC again seen with unchanged position in the mid to distal SVC.  No evidence of pneumonia or pleural effusions.

Image dtype : torch.float32
